# Overlapping Subsequence Augmentation with Poly-N Padding

This notebook generates augmented 40-nt overlapping subsequences from original genomic sequences for use as input to the Aggregated CNN–LSTM–Attention–Residual model.

**Pipeline:**

1. Load original sequences from a FASTA file.
2. Pad each sequence with 40-nt poly-N at both ends.
3. Generate all valid 40-nt subsequences using variable overlaps (5–20 nt) and a 15-consecutive-nucleotide commonality criterion.
4. Write unique subsequences to a CSV file (one row per subsequence).

Run this notebook **before** the main model notebook. Each class (species/group) requires its own FASTA input file and produces its own CSV output file.

**Repository:** https://github.com/parkingvarsson/Aggregated-DL

## 1. Dependencies

In [ ]:
# Biopython is required for FASTA parsing.
# Uncomment the line below if running on Google Colab for the first time.
# !pip install biopython -q

from Bio import SeqIO


## 2. Core Augmentation Functions

### 2.1 Consecutive-Nucleotide Commonality Check

Returns `True` if two sequences share at least 15 consecutive nucleotides. Used to ensure that newly generated subsequences maintain biological continuity with previously extracted windows.

In [ ]:
def has_15_consecutive_common(seq1, seq2):
    """
    Check if two sequences share at least 15 consecutive nucleotides.
    """
    for i in range(len(seq1) - 14):  # windows of 15 nucleotides
        if seq1[i:i+15] in seq2:
            return True
    return False


### 2.2 Variable-Overlap Subsequence Generator

Generates all unique 40-nt subsequences from a padded sequence using two strategies:

- **Variable overlaps (5–20 nt):** sliding windows with left, right, and bilateral overlap.
- **15-nt commonality:** additional windows sharing ≥15 consecutive nucleotides with any already-selected window.

Uniqueness is enforced via a Python `set`.

In [ ]:
def generate_sequences_with_variable_overlap_and_common_bases(
        seq, k=40, min_overlap=5, max_overlap=20):
    """
    Generate all possible k-mers (subsequences) of length k (40 nt) from the
    sequence using:
      1. Overlaps of varying lengths (5-20 nt).
      2. Sequences sharing >=15 consecutive nucleotides with a previously
         extracted sequence.
    """
    valid_sequences = set()  # set ensures uniqueness

    # Strategy 1: variable overlaps (5 to 20 nucleotides)
    for overlap in range(min_overlap, max_overlap + 1):
        # Left overlap
        for i in range(0, len(seq) - k + 1, k - overlap):
            valid_sequences.add(str(seq[i:i + k]))
        # Right overlap
        for i in range(overlap, len(seq) - k + 1, k - overlap):
            valid_sequences.add(str(seq[i:i + k]))
        # Bilateral overlap
        for i in range(overlap // 2, len(seq) - k + 1, k - overlap):
            valid_sequences.add(str(seq[i:i + k]))

    # Strategy 2: 15-consecutive-nucleotide commonality
    for i in range(len(seq) - k + 1):
        sub_seq = str(seq[i:i + k])
        if not valid_sequences:
            valid_sequences.add(sub_seq)
        else:
            if any(has_15_consecutive_common(existing, sub_seq)
                   for existing in valid_sequences):
                valid_sequences.add(sub_seq)

    return list(valid_sequences)


### 2.3 FASTA Processing Pipeline

For each sequence in the input FASTA file:

1. Prepend and append 40-nt poly-N padding.
2. Generate augmented subsequences.
3. Write each unique 40-nt subsequence to the output CSV as `sequence, label`.

The label is extracted from the FASTA record ID (prefix before the first `_`).

In [ ]:
def process_fasta_for_nn(input_fasta, output_file, k=40,
                         min_overlap=5, max_overlap=20):
    """
    Process each sequence in the input FASTA file to generate augmented
    40-nt subsequences with variable overlaps and poly-N padding.

    Parameters
    ----------
    input_fasta  : str  Path to the input FASTA file.
    output_file  : str  Path for the output CSV file.
    k            : int  Subsequence length (default 40 nt).
    min_overlap  : int  Minimum overlap between consecutive windows (default 5).
    max_overlap  : int  Maximum overlap between consecutive windows (default 20).
    """
    sequence_counts   = []
    total_augmented   = 0
    original_lengths  = []
    augmented_lengths = []

    with open(output_file, 'w') as output_handle:
        for record in SeqIO.parse(input_fasta, 'fasta'):
            original_lengths.append(len(record.seq))

            # Add poly-N padding of length k to both ends
            padded_seq = 'N' * k + str(record.seq) + 'N' * k

            # Generate augmented subsequences
            overlap_sequences = \
                generate_sequences_with_variable_overlap_and_common_bases(
                    padded_seq, k, min_overlap, max_overlap)

            all_sequences = set(overlap_sequences)  # enforce uniqueness
            count = len(all_sequences)
            sequence_counts.append(count)
            total_augmented += count
            augmented_lengths.extend([len(seq) for seq in all_sequences])

            label = record.id.split('_')[0]  # label = prefix before first '_'
            for sub_seq in all_sequences:
                output_handle.write(f'{sub_seq}, {label}\n')

    # ── Summary statistics ────────────────────────────────────────────────
    print('\nTotal:')
    print(f'The number of original sequences: {len(sequence_counts)}')

    if original_lengths:
        avg_orig = sum(original_lengths) // len(original_lengths)
        print(f'The length of original sequences: {avg_orig} nucleotides (average)')

    if augmented_lengths:
        unique_aug_lengths = set(augmented_lengths)
        if len(unique_aug_lengths) == 1:
            print(f'The length of augmented sequences: '
                  f'{unique_aug_lengths.pop()} nucleotides (all same length)')
        else:
            avg_aug = sum(augmented_lengths) // len(augmented_lengths)
            print(f'The length of augmented sequences: '
                  f'{avg_aug} nucleotides (average)')
            print(f'Note: Multiple lengths found: {unique_aug_lengths}')

    if sequence_counts:
        avg_aug_per_seq = total_augmented // len(sequence_counts)
        print(f'The number of augmented sequences for each sequence: '
              f'{avg_aug_per_seq}')

    print(f'The number of total augmented sequences: {total_augmented}')


## 3. Run Augmentation

Update `input_fasta` and `output_file` to your actual Google Drive paths. Run this cell once per class (FASTA file). The output CSV is used directly as input to the main model notebook.

In [ ]:
# ── Paths — update before running ────────────────────────────────────────────
input_fasta = '/content/drive/MyDrive/.../Plant without SDs_200nt.fasta'
output_file = '/content/drive/MyDrive/.../Plant without SDs.csv'

# ── Run augmentation ──────────────────────────────────────────────────────────
process_fasta_for_nn(input_fasta, output_file)
